# 05 - Advanced Models

In [ ]:
import pandas as pd
import geopandas as gpd
from itertools import product
from xgboost import XGBRegressor
from sklearn.metrics import r2_score, mean_absolute_error, root_mean_squared_error

In [ ]:
# Load cleaned data
df = pd.read_csv("cleaned.csv", dtype={"CloseMonth": str})
months = sorted(df["CloseMonth"].unique())
df.shape

## Features

In [ ]:
# Same features as 05_feature_engineering
df["BedBathRatio"] = (df["BedroomsTotal"] / df["BathroomsTotalInteger"]).replace(float("inf"), 0).fillna(0)
df["Age"] = df["CloseMonth"].str[:4].astype(int) - df["YearBuilt"]

districts = gpd.read_file("school_districts/DistrictAreas2425.shp")
points = gpd.GeoDataFrame(geometry=gpd.points_from_xy(df["Longitude"], df["Latitude"]), index=df.index, crs="EPSG:4326")
points = points.to_crs(districts.crs)
joined = gpd.sjoin(points, districts[["DistrictNa", "DistrictTy", "geometry"]], how="left", predicate="within")

# Homes in both an elementary and a high school district: keep the unified district if there is one, otherwise the elementary one
joined["priority"] = joined["DistrictTy"].map({"Unified": 0, "Elementary": 1, "High": 2})
joined = joined.sort_values("priority").groupby(level=0).first()
df["SchoolDistrict"] = joined["DistrictNa"].fillna("None")

df = pd.get_dummies(df, columns=["SchoolDistrict"], drop_first=True, dtype=int)
features = [c for c in df.columns if c not in ["ClosePrice", "CloseMonth"]]
df.shape

## Train/test split

In [ ]:
# Test on the most recent month, train on the X months before it
X = 4
train = df[df["CloseMonth"].isin(months[-1 - X:-1])]
test = df[df["CloseMonth"] == months[-1]]
train.shape, test.shape

## Tuning

In [ ]:
# Tune on the last training month so the test month stays untouched
fit_part = train[train["CloseMonth"] != months[-2]]
val_part = train[train["CloseMonth"] == months[-2]]

grid = product([4, 6, 8], [0.05, 0.1], [300, 600])

tuning = []
for depth, lr, n in grid:
    params = {"max_depth": depth, "learning_rate": lr, "n_estimators": n}
    model = XGBRegressor(**params, random_state=42, n_jobs=-1)
    model.fit(fit_part[features], fit_part["ClosePrice"])
    val_r2 = r2_score(val_part["ClosePrice"], model.predict(val_part[features]))
    tuning.append((val_r2, params))
    print(f"depth {depth}, learning rate {lr}, trees {n}: validation R2 {val_r2:.3f}")

In [ ]:
best_r2, best_params = max(tuning, key=lambda t: t[0])
print(f"Best settings: {best_params}")

## Results

In [ ]:
models = {
    "XGBoost (default settings)": XGBRegressor(random_state=42, n_jobs=-1),
    "XGBoost (tuned)": XGBRegressor(**best_params, random_state=42, n_jobs=-1),
}

for name, model in models.items():
    model.fit(train[features], train["ClosePrice"])
    pred = model.predict(test[features])
    print(name)
    print(f"Test R2: {r2_score(test['ClosePrice'], pred):.3f}")
    print(f"Test MAE: ${mean_absolute_error(test['ClosePrice'], pred):,.0f}")
    print(f"Test RMSE: ${root_mean_squared_error(test['ClosePrice'], pred):,.0f}")
    print()